In [ ]:
%%sql -r dataframe_1
USE ROLE CHEETAH_ROLE;
USE WAREHOUSE CHEETAH_WH;
USE DATABASE CHEETAH_DB;
USE SCHEMA GOLD_BUSINESS_DATA;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE STAGE pdf_stage
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
# Upload the PDF to the stage using Snowpark session
from snowflake.snowpark.context import get_active_session
import os

session = get_active_session()

# Find the PDF file in the workspace
pdf_path = '/workspace/2024-Annual-Report-Target-Corporation.pdf'
if not os.path.exists(pdf_path):
    # Try alternate location
    for root, dirs, files in os.walk('/workspace'):
        for f in files:
            if f.endswith('.pdf'):
                pdf_path = os.path.join(root, f)
                break

print(f"Uploading: {pdf_path}")
result = session.file.put(pdf_path, '@CHEETAH_DB.GOLD_BUSINESS_DATA.pdf_stage', auto_compress=False, overwrite=True)
for r in result:
    print(f"  {r.source} -> {r.target} [{r.status}]")

In [ ]:
%%sql -r dataframe_3
SELECT 
    SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
        TO_FILE('@CHEETAH_DB.GOLD_BUSINESS_DATA.pdf_stage', '2024-Annual-Report-Target-Corporation.pdf'),
        OBJECT_CONSTRUCT('mode', 'LAYOUT')
    ) AS parsed_content;

In [ ]:
import json

parsed_pdf_pd = dataframe_3.to_pandas()
parsed_json = json.loads(parsed_pdf_pd.iloc[0]['PARSED_CONTENT'])
full_text = parsed_json.get('content', '')
print(f"Extracted {len(full_text)} characters from PDF")
print(f"First 500 chars:\n{full_text[:500]}")

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text)
print(f"Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i} ({len(chunk)} chars): {chunk[:100]}...")

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

chunks_df = pd.DataFrame({
    'CHUNK_ID': range(len(chunks)),
    'CHUNK_TEXT': chunks,
    'SOURCE_FILE': '2024-Annual-Report-Target-Corporation.pdf'
})

snow_df = session.create_dataframe(chunks_df)
snow_df.write.mode('overwrite').save_as_table('TARGET_PDF_CHUNKS')
print(f"Wrote {len(chunks)} chunks to CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_CHUNKS")

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_EMBEDDINGS AS
SELECT
    CHUNK_ID,
    CHUNK_TEXT,
    SOURCE_FILE,
    SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', CHUNK_TEXT) AS EMBEDDING
FROM CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_CHUNKS;

In [ ]:
%%sql -r dataframe_5
SELECT 
    CHUNK_ID,
    LEFT(CHUNK_TEXT, 80) AS CHUNK_PREVIEW,
    EMBEDDING::ARRAY AS EMBEDDING_VECTOR
FROM CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_EMBEDDINGS
ORDER BY CHUNK_ID
LIMIT 5;

In [ ]:
%%sql -r dataframe_6
SELECT COUNT(*) AS ROW_COUNT FROM TARGET_PDF_EMBEDDINGS;

In [ ]:
%%sql -r comment_result
COMMENT ON TABLE CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_EMBEDDINGS IS
  'Chunked and embedded text from the Target Corporation 2024 Annual Report PDF. Each row is a text chunk (approx 1000 chars with 200-char overlap) with its e5-base-v2 vector embedding. Use for semantic search and RAG over Target financial disclosures, performance highlights, and strategic outlook.';
COMMENT ON COLUMN CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_EMBEDDINGS.CHUNK_ID IS
  'Sequential zero-based integer identifying the chunk position within the source document.';
COMMENT ON COLUMN CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_EMBEDDINGS.CHUNK_TEXT IS
  'Raw text content of the chunk extracted from the PDF layout parse. Approximately 1000 characters with 200-character overlap between consecutive chunks.';
COMMENT ON COLUMN CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_EMBEDDINGS.SOURCE_FILE IS
  'Filename of the original PDF stored in the pdf_stage from which this chunk was extracted.';
COMMENT ON COLUMN CHEETAH_DB.GOLD_BUSINESS_DATA.TARGET_PDF_EMBEDDINGS.EMBEDDING IS
  'e5-base-v2 768-dimensional vector embedding of CHUNK_TEXT, generated via SNOWFLAKE.CORTEX.EMBED_TEXT_768. Use with VECTOR_COSINE_SIMILARITY for semantic search.';